# theLook eCommerce

Prepare customers, orders, and order items in BigQuery, then export them to `data/thelook/` for Tableau.

Source: `bigquery-public-data.thelook_ecommerce`. Run this notebook from the repository folder in VS Code. Running the three table cells replaces the matching BigQuery tables.


## 1. Connect

Choose a Python kernel and run this cell. Sign in with the Google account that has access to `myproject4142-512119`.


In [ ]:
%pip install -q google-cloud-bigquery pandas db-dtypes pydata-google-auth

from pathlib import Path
from google.cloud import bigquery
import pydata_google_auth

credentials, _ = pydata_google_auth.default(
    scopes=["https://www.googleapis.com/auth/bigquery"]
)
client = bigquery.Client(
    project="myproject4142-512119",
    credentials=credentials,
    location="US"
)

data_folder = Path("data/thelook")
data_folder.mkdir(parents=True, exist_ok=True)
print("Connected. CSV files will be saved in:", data_folder.resolve())


## 2. dim_customers

One row per registered customer, including customers with no orders.


In [ ]:
query = """
CREATE OR REPLACE TABLE `myproject4142-512119.thelook_analysis.dim_customers` AS
SELECT
  id AS user_id,
  DATE(created_at) AS signup_date,
  country,
  state,
  city,
  traffic_source,
  age,
  gender
FROM `bigquery-public-data.thelook_ecommerce.users`;
"""
client.query(query).result()

dim_customers = client.query(
    "SELECT * FROM `myproject4142-512119.thelook_analysis.dim_customers`"
).to_dataframe(create_bqstorage_client=False)

dim_customers.to_csv(data_folder / "dim_customers.csv", index=False)
print(f"Saved {len(dim_customers):,} rows to dim_customers.csv")
dim_customers.head()


## 3. fact_orders

One row per order. Keep every status so Tableau can show completed, cancelled, returned, and in-progress orders.


In [ ]:
query = """
CREATE OR REPLACE TABLE `myproject4142-512119.thelook_analysis.fact_orders` AS
SELECT
  order_id,
  user_id,
  DATE(created_at) AS order_date,
  status AS order_status,
  num_of_item AS item_count,
  DATE(shipped_at) AS shipped_date,
  DATE(delivered_at) AS delivered_date,
  DATE(returned_at) AS returned_date
FROM `bigquery-public-data.thelook_ecommerce.orders`;
"""
client.query(query).result()

fact_orders = client.query(
    "SELECT * FROM `myproject4142-512119.thelook_analysis.fact_orders`"
).to_dataframe(create_bqstorage_client=False)

fact_orders.to_csv(data_folder / "fact_orders.csv", index=False)
print(f"Saved {len(fact_orders):,} rows to fact_orders.csv")
fact_orders.head()


## 4. fact_order_items

One row per item in an order. Add product details and inventory cost. Gross profit is sale price minus item cost.


In [ ]:
query = """
CREATE OR REPLACE TABLE `myproject4142-512119.thelook_analysis.fact_order_items` AS
SELECT
  oi.id AS order_item_id,
  oi.order_id,
  oi.user_id,
  oi.product_id,
  DATE(oi.created_at) AS item_date,
  oi.status AS item_status,
  DATE(oi.returned_at) AS returned_date,
  p.name AS product_name,
  p.category,
  p.department,
  p.brand,
  oi.sale_price,
  i.cost AS unit_cost,
  oi.sale_price - i.cost AS gross_profit
FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
LEFT JOIN `bigquery-public-data.thelook_ecommerce.products` p
  ON oi.product_id = p.id
LEFT JOIN `bigquery-public-data.thelook_ecommerce.inventory_items` i
  ON oi.inventory_item_id = i.id;
"""
client.query(query).result()

fact_order_items = client.query(
    "SELECT * FROM `myproject4142-512119.thelook_analysis.fact_order_items`"
).to_dataframe(create_bqstorage_client=False)

fact_order_items.to_csv(data_folder / "fact_order_items.csv", index=False)
print(f"Saved {len(fact_order_items):,} rows to fact_order_items.csv")
fact_order_items.head()
